# Comparative Sentiment Analysis: Indonesian E-Commerce Reviews (Scopus Pipeline)
## Notebook 03: Baseline Tree-Based Models (Random Forest & XGBoost + TF-IDF)

```text
Author: Muhammad Farrel Haidar
Project: AI PR & KOL Specialist DSS
Date: 2026-10-14
Study: Comparative Sentiment Analysis (IndoBERT vs Tree-Based Models)
Models: Random Forest (300 Trees) & XGBoost (Multi:softprob, Early Stopping)
```

---
### 1. Landasan Teoretis & Rumus Matematis (Bahan Manuskrip)

#### a. Term Frequency – Inverse Document Frequency (TF-IDF)
Ekstraksi fitur teks konvensional menggunakan bobot TF-IDF pada rentang n-gram (unigram & bigram):

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \log\left(\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|}\right) + 1$$

di mana $\text{TF}(t,d)$ adalah frekuensi kemunculan term $t$ dalam dokumen $d$, dan $|D|$ adalah total dokumen dalam korpus latih.

#### b. Metrik Evaluasi Klasifikasi Multi-Kelas
Untuk menangani ketidakseimbangan kelas (*class imbalance*), evaluasi menggunakan rata-rata makro (*Macro-Averaged*) dan berbobot (*Weighted-Averaged*):

$$\text{Precision}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FP}_c}, \quad \text{Recall}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FN}_c}$$

$$\text{F1-Score}_c = 2 \times \frac{\text{Precision}_c \times \text{Recall}_c}{\text{Precision}_c + \text{Recall}_c}$$

$$\text{Macro-F1} = \frac{1}{|C|} \sum_{c \in C} \text{F1-Score}_c$$

$$\text{Weighted-F1} = \sum_{c \in C} \frac{N_c}{N} \times \text{F1-Score}_c$$

di mana $|C| = 3$ (Negatif, Netral, Positif) dan $N_c$ adalah jumlah sampel kelas $c$ pada *test set*.

In [ ]:
# Author: Muhammad Farrel Haidar
# Project: AI PR & KOL Specialist DSS
# Date: 2026-10-14

import os
import time
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (6, 5)
print("Libraries loaded successfully.")

### 2. Load Partisi Data Latih, Validasi, dan Uji

In [ ]:
splits_dir = "../data/splits" if os.path.exists("../data/splits") else "data/splits"
train_df = pd.read_csv(os.path.join(splits_dir, "train.csv"))
val_df = pd.read_csv(os.path.join(splits_dir, "val.csv"))
test_df = pd.read_csv(os.path.join(splits_dir, "test.csv"))

print(f"Train size : {len(train_df):,} baris")
print(f"Val size   : {len(val_df):,} baris")
print(f"Test size  : {len(test_df):,} baris")

### 3. Vektorisasi TF-IDF (Unigram + Bigram)

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    min_df=2,
    max_df=0.95
)

X_train = vectorizer.fit_transform(train_df["cleaned_text"].fillna(""))
X_val = vectorizer.transform(val_df["cleaned_text"].fillna(""))
X_test = vectorizer.transform(test_df["cleaned_text"].fillna(""))

y_train = train_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values

print(f"Vocabulary size: {len(vectorizer.vocabulary_):,} features")
print(f"X_train shape  : {X_train.shape}")

### 4. Pelatihan & Evaluasi Model 1: Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
rf.fit(X_train, y_train)
rf_time = time.time() - t0
print(f"Training time: {rf_time:.2f} detik")

# Evaluasi pada Test Set
rf_pred = rf.predict(X_test)
print("\n--- Classification Report: Random Forest ---")
print(classification_report(y_test, rf_pred, target_names=["Negatif", "Netral", "Positif"]))

# Confusion Matrix Plot
cm_rf = confusion_matrix(y_test, rf_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=["Negatif", "Netral", "Positif"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix: Random Forest + TF-IDF")
plt.show()

### 5. Pelatihan & Evaluasi Model 2: XGBoost Classifier (Early Stopping)

In [ ]:
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=3,
    random_state=42,
    eval_metric="mlogloss",
    early_stopping_rounds=20
)

t0 = time.time()
xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
xgb_time = time.time() - t0
print(f"Training time: {xgb_time:.2f} detik (Best Iteration: {getattr(xgb, 'best_iteration', 'N/A')})")

# Evaluasi pada Test Set
xgb_pred = xgb.predict(X_test)
print("\n--- Classification Report: XGBoost ---")
print(classification_report(y_test, xgb_pred, target_names=["Negatif", "Netral", "Positif"]))

# Confusion Matrix Plot
cm_xgb = confusion_matrix(y_test, xgb_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_xgb, display_labels=["Negatif", "Netral", "Positif"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix: XGBoost + TF-IDF")
plt.show()

### 6. Tabel Komparasi Kinerja Baseline (Table 2 Preview)

In [ ]:
m_rf = pd.read_csv("../data/results/metrics_rf.csv" if os.path.exists("../data/results/metrics_rf.csv") else "data/results/metrics_rf.csv")
m_xgb = pd.read_csv("../data/results/metrics_xgb.csv" if os.path.exists("../data/results/metrics_xgb.csv") else "data/results/metrics_xgb.csv")

summary = pd.concat([m_rf, m_xgb], ignore_index=True)
cols = ["Model", "Accuracy", "Precision_Macro", "Recall_Macro", "F1_Macro", "F1_Weighted", "ROC_AUC_OVR", "Training_Time_Sec"]
display(summary[cols])